# Language family vs. cultural zone

**Phase I, AI-aided pair.** Do patterns in the double-centered caption-language ×
image-locale heatmap align with language family or with the chosen Inglehart–Welzel cultural zones?

Checkpoint: `google/siglip2-base-patch16-256`, revision
`3f9f96cb90da5dbc758b01813f2f6f1aee24c1ab`.
Each matrix value is an averaged margin after removing language and locale baselines.
Positive values indicate above-baseline association. Cultural-zone assignments are
exploratory locale-level groupings, not per-image annotations.

In [1]:
import sys
from pathlib import Path

# Locate the AI-aided folder from the repo root, notebook folder, or its parent.
candidates = [Path.cwd(), *Path.cwd().parents]
AI_DIR = next(p for base in candidates for p in [base, base / "phase-1" / "ai-aided"]
              if (p / "implementation" / "config.py").is_file())
if str(AI_DIR) not in sys.path:
    sys.path.insert(0, str(AI_DIR))

import numpy as np
import pandas as pd
from IPython.display import display

from implementation import config
from implementation.artifacts import read_metadata, score_settings

read_metadata(config.SCORE_METADATA, "-m implementation.compute_scores", score_settings())

langs = config.active_languages()
m = pd.read_parquet(config.MATRICES_FILE)
D = (m[m.state == "double"]
     .pivot(index="caption_lang", columns="locale", values="value")
     .reindex(index=langs, columns=langs))
family = {c: g for g, cs in config.LANGUAGE_FAMILIES.items() for c in cs}
zone = {c: g for g, cs in config.CULTURAL_ZONES.items() for c in cs}
print(f"{len(langs)} languages, {len(langs) * (len(langs) - 1)} off-diagonal pairs")

36 languages, 1260 off-diagonal pairs


## 1. Classify every off-diagonal pair

In [2]:
CATS = ["same family, same zone", "same family, different zone",
        "different family, same zone", "different family, different zone"]

rows = []
for a in langs:              # a = caption language
    for b in langs:          # b = image locale
        if a == b:
            continue
        same_f, same_z = family[a] == family[b], zone[a] == zone[b]
        cat = CATS[[ (True, True), (True, False), (False, True), (False, False) ].index((same_f, same_z))]
        rows.append({"caption_lang": a, "locale": b, "pair": f"{a}->{b}",
                     "category": cat, "residual": D.at[a, b]})
pairs = pd.DataFrame(rows)
pairs.head()

,caption_lang,locale,pair,category,residual
0,ar,bn,ar->bn,"different family, same zone",0.007218
1,ar,cs,ar->cs,"different family, different zone",-0.007795
2,ar,da,ar->da,"different family, different zone",-0.003683
3,ar,de,ar->de,"different family, different zone",0.000937
4,ar,el,ar->el,"different family, different zone",0.001338


## 2. The disagreement table

Mean residual per category with a 95 % bootstrap confidence interval (resampling pairs).
*Caveat:* pairs are not fully independent (the same language appears in many pairs), so the intervals are
somewhat optimistic. Treat them as a guide rather than a formal test.

In [3]:
rng = np.random.default_rng(0)

def boot_ci(x, n=5000):
    means = rng.choice(x, size=(n, len(x)), replace=True).mean(axis=1)
    return np.percentile(means, [2.5, 97.5])

diag = np.diag(D.to_numpy())
lo, hi = boot_ci(diag)
table = [{"category": "matched (diagonal)", "n_pairs": len(diag), "mean": diag.mean(),
          "ci_low": lo, "ci_high": hi}]
for c in CATS:
    x = pairs.loc[pairs.category == c, "residual"].to_numpy()
    lo, hi = boot_ci(x)
    table.append({"category": c, "n_pairs": len(x), "mean": x.mean(), "ci_low": lo, "ci_high": hi})
table = pd.DataFrame(table).set_index("category")
table.round(4)

,n_pairs,mean,ci_low,ci_high
category,,,,
matched (diagonal),36,0.0133,0.0108,0.0158
"same family, same zone",42,0.0062,0.0045,0.0080
"same family, different zone",44,0.0035,0.0021,0.0049
"different family, same zone",116,0.0042,0.0031,0.0054
"different family, different zone",1058,-0.0013,-0.0016,-0.0010


### Interpreting the group comparison

Compare **same family, different zone** with **different family, same zone**.
The groupings overlap. Pairs sharing both labels cannot distinguish the two explanations.
The pair-bootstrap intervals are descriptive because pairs share languages and locales.

## 3. Which pairs drive the two disagreement categories?

### Same family, different zone

Eight highest and eight lowest residuals.

In [4]:
sub = pairs[pairs.category == "same family, different zone"].sort_values("residual", ascending=False)
display(pd.concat([sub.head(8), sub.tail(8)])[["pair", "residual"]].round(4).reset_index(drop=True))

,pair,residual
0,hi->bn,0.0160
1,mi->id,0.0127
2,ar->he,0.0105
3,he->ar,0.0103
4,ru->cs,0.0085
5,ro->pt,0.0081
6,hr->ru,0.0081
7,mi->fil,0.0079
8,fa->hi,-0.0021
9,es->ro,-0.0025


### Different family, same zone

Eight highest and eight lowest residuals.

In [5]:
sub = pairs[pairs.category == "different family, same zone"].sort_values("residual", ascending=False)
display(pd.concat([sub.head(8), sub.tail(8)])[["pair", "residual"]].round(4).reset_index(drop=True))

,pair,residual
0,hi->te,0.0299
1,th->vi,0.0212
2,fi->sv,0.0189
3,no->fi,0.0167
4,ko->ja,0.0154
5,te->vi,0.0139
6,da->fi,0.0134
7,fi->de,0.0130
8,el->uk,-0.0046
9,es->cs,-0.0051


### Interpretation limits

Top and bottom pairs expose variation within each category. A group average can conceal
reversals and asymmetric associations between language and locale. Caption content and image selection
remain alternative explanations for residual differences.

## 4. Limitations

- Cultural zones assume country associations for locale labels. They are not measured per image.
- Hindi-locale images were sampled globally.
- Pair-bootstrap intervals omit dependence from repeated languages and locales.
- Caption content and image selection can contribute to the differences.

The [main experiment notebook](../Phase_I_Experiments.ipynb) contains the heatmap,
image-level matching inference, and concrete retrieval examples.